# 02 — Exploratory Analysis

This notebook explores the transaction data visually to identify
patterns before running formal statistical tests.

**What this notebook does:**
- Loads the clean datasets from `data/`
- Visualizes transaction distribution by time slot and day of week
- Explores the most frequent category pairs
- Identifies patterns that will guide the statistical analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load datasets
df = pd.read_csv(Path("../data/transactions_clean.csv"))
pairs_df = pd.read_csv(Path("../data/category_pairs.csv"))

print(f"Transactions: {df.shape}")
print(f"Category pairs: {pairs_df.shape}")
df.head()

## Temporal distribution

In [ ]:
# Transactions by time slot and day of week
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

slot_order = ["Morning", "Afternoon", "Evening"]
sns.countplot(data=df, x="time_slot", order=slot_order, palette="Oranges", ax=axes[0])
axes[0].set_title("Transactions by Time Slot")
axes[0].set_xlabel("")

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sns.countplot(data=df, x="day_of_week", order=day_order, palette="Blues", ax=axes[1])
axes[1].set_title("Transactions by Day of Week")
axes[1].set_xlabel("")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("../outputs/figures/01_temporal_distribution.png", dpi=150)
plt.show()

## Most frequent category pairs

In [ ]:
# Top 15 most frequent category pairs overall
pairs_df["pair"] = pairs_df["category_a"] + " + " + pairs_df["category_b"]
top_pairs = pairs_df["pair"].value_counts().head(15).reset_index()
top_pairs.columns = ["pair", "count"]

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=top_pairs, y="pair", x="count", palette="Blues_r", ax=ax)
ax.set_title("Top 15 Most Frequent Category Pairs", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of co-purchases")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("../outputs/figures/02_top_category_pairs.png", dpi=150)
plt.show()

## Category pair frequency by time slot

In [ ]:
# Heatmap: category_a vs category_b (overall co-purchase frequency)
pivot = pairs_df.groupby(["category_a", "category_b"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5, ax=ax)
ax.set_title("Co-purchase Frequency: Category Pairs", fontsize=13, fontweight="bold")
ax.set_ylabel("")
ax.set_xlabel("")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../outputs/figures/03_copurchase_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Faceted: top pairs per time slot
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)

for ax, slot in zip(axes, ["Morning", "Afternoon", "Evening"]):
    slot_data = pairs_df[pairs_df["time_slot"] == slot]
    top = slot_data["pair"].value_counts().head(8).reset_index()
    top.columns = ["pair", "count"]
    sns.barplot(data=top, y="pair", x="count", palette="Oranges_r", ax=ax)
    ax.set_title(f"{slot}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Co-purchases")
    ax.set_ylabel("")

plt.suptitle("Top Category Pairs by Time Slot", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../outputs/figures/04_pairs_by_timeslot.png", dpi=150, bbox_inches="tight")
plt.show()

## Category pair frequency by day of week

In [ ]:
# Heatmap: time slot vs pair count
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot_day = pairs_df.groupby(["day_of_week", "pair"]).size().unstack(fill_value=0)
pivot_day = pivot_day.reindex(day_order)

# Show only top 10 pairs
top10 = pairs_df["pair"].value_counts().head(10).index
pivot_day = pivot_day[top10]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot_day, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5, ax=ax)
ax.set_title("Top 10 Pairs — Frequency by Day of Week", fontsize=13, fontweight="bold")
ax.set_ylabel("")
ax.set_xlabel("")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../outputs/figures/05_pairs_by_dayofweek.png", dpi=150)
plt.show()

## Summary

- Afternoon is the peak shopping time slot
- Day of week distribution is relatively uniform
- Top category pairs show clear temporal variation across time slots
- Visual patterns confirm that temporal segmentation is meaningful
- Ready for formal statistical testing in `03_statistical_analysis.ipynb`